# 04 - Preprocessing Citra & Pembuatan DataLoader PyTorch
### Proyek CITRA SERUPA — Sistem Rekomendasi Gambar Fashion Berdasarkan Kemiripan Visual

---

**Tujuan Notebook:**
Model Deep Learning (seperti ResNet-50) tidak bisa langsung membaca file gambar mentah. Citra harus dipersiapkan melalui pipeline **Preprocessing**:
1. **Resize & Center Crop (224x224):** Menyamakan dimensi spasial citra ke standar arsitektur neural network.
2. **Konversi ke Tensor (`ToTensor`):** Mengubah format gambar dari nilai piksel integer [0, 255] menjadi matriks float berdimensi `[Channel, Height, Width]` dengan rentang [0.0, 1.0].
3. **Normalisasi Standar ImageNet (`Normalize`):** Menyesuaikan distribusi warna piksel menggunakan rata-rata (*mean*) dan deviasi standar (*std*) ImageNet:
   - Mean: `[0.485, 0.456, 0.406]`
   - Std: `[0.229, 0.224, 0.225]`
4. **Custom PyTorch Dataset (`FashionDataset`):** Menggabungkan pembacaan gambar dari Google Drive dengan tabel indeks `metadata.csv`.
5. **PyTorch DataLoader:** Memuat gambar secara efisien dalam kelompok (*batch*) dengan akselerasi GPU T4.

## 1. Hubungkan Google Colab ke Google Drive & Verifikasi Direktori

In [ ]:
from google.colab import drive
import os
import torch
import pandas as pd

# 1. Hubungkan Google Drive
drive.mount('/content/drive')

# 2. Deteksi lokasi folder dataset
base_dir = '/content/drive/MyDrive/CITRA-SERUPA'
if not os.path.exists(base_dir):
    base_dir = '/content/drive/MyDrive/CITRA_SERUPA'

dataset_dir = os.path.join(base_dir, 'dataset')
images_dir = os.path.join(dataset_dir, 'images')
csv_path = os.path.join(dataset_dir, 'metadata.csv')

# Cek ketersediaan GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 60)
print(f"⚡ Device yang Digunakan : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"📁 Folder Gambar        : {images_dir}")
print(f"📄 File Metadata        : {csv_path}")
print("=" * 60)

# Baca metadata
df_meta = pd.read_csv(csv_path)
print(f"✅ Metadata berhasil dimuat: {len(df_meta):,} gambar terdaftar.")

## 2. Mendefinisikan Pipeline Transformasi Preprocessing

In [ ]:
from torchvision import transforms

# Definisi pipeline preprocessing resmi standar ImageNet / ResNet
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),           # 1. Menyamakan ukuran piksel
    transforms.ToTensor(),                   # 2. Konversi PIL Image ke Tensor [0.0, 1.0]
    transforms.Normalize(                    # 3. Normalisasi distribusi warna ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Pipeline Preprocessing berhasil didefinisikan:")
for i, step in enumerate(transform_pipeline.transforms, 1):
    print(f"   {i}. {step.__class__.__name__}")

## 3. Pembuatan Custom Dataset PyTorch (`FashionDataset`)

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class FashionDataset(Dataset):
    """
    Custom Dataset PyTorch untuk memuat citra fashion beserta label produknya.
    """
    def __init__(self, metadata_df, img_dir, transform=None):
        self.df = metadata_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image_name']
        item_id = row['item_id']
        img_path = os.path.join(self.img_dir, img_name)

        # Buka gambar dan pastikan format RGB
        image = Image.open(img_path)
        if image.mode != 'RGB':
            image = image.convert('RGB')

        # Terapkan transformasi preprocessing
        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = transforms.ToTensor()(image)

        return image_tensor, img_name, item_id

# Inisialisasi dataset
fashion_dataset = FashionDataset(
    metadata_df=df_meta,
    img_dir=images_dir,
    transform=transform_pipeline
)

print(f"✅ FashionDataset berhasil dibuat dengan total {len(fashion_dataset):,} sampel data.")

# Uji coba ambil 1 sampel data pertama
sample_tensor, sample_name, sample_item = fashion_dataset[0]
print(f"\n🔍 Contoh Sampel Data Pertama:")
print(f" - Nama File   : {sample_name}")
print(f" - Item ID     : {sample_item}")
print(f" - Bentuk Tensor: {sample_tensor.shape} (Format: [Channel, Height, Width])")
print(f" - Rentang Nilai: Min = {sample_tensor.min():.2f}, Max = {sample_tensor.max():.2f}")

## 4. Pembuatan PyTorch DataLoader (Batching & Pengujian Batch)

In [ ]:
from torch.utils.data import DataLoader

# Tentukan ukuran batch (batch_size = 64 sangat optimal untuk GPU T4 Colab)
BATCH_SIZE = 64

fashion_loader = DataLoader(
    dataset=fashion_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # Tidak perlu diacak agar urutan indeks konsisten untuk ekstraksi fitur
    num_workers=2,       # Mempercepat pembacaan data secara paralel
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✅ DataLoader berhasil dibuat!")
print(f" - Ukuran Batch (Batch Size) : {BATCH_SIZE}")
print(f" - Total Batch               : {len(fashion_loader)} batch")

# Uji coba memuat 1 batch ke GPU
batch_images, batch_names, batch_items = next(iter(fashion_loader))
batch_images = batch_images.to(device)

print(f"\n🚀 Pengujian Batch Pertama pada GPU:")
print(f" - Dimensi Batch Tensor : {batch_images.shape} ([Batch_Size, Channels, Height, Width])")
print(f" - Lokasi Memori Tensor : {batch_images.device}")
print(f" - Jumlah Item Terbaca  : {len(batch_names)} gambar")

## 5. Visualisasi Perbandingan: Citra Asli vs Citra Pasca-Preprocessing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def denormalize(tensor):
    """
    Mengembalikan tensor yang telah dinormalisasi ImageNet kembali ke citra visual RGB [0, 1].
    """
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

# Pilih 3 sampel acak untuk divisualisasikan
fig, axes = plt.subplots(3, 2, figsize=(8, 10))
indices = [0, len(df_meta)//2, len(df_meta)-1]

for row_idx, data_idx in enumerate(indices):
    row = df_meta.iloc[data_idx]
    img_path = os.path.join(images_dir, row['image_name'])
    
    # 1. Citra Asli (PIL)
    raw_img = Image.open(img_path)
    axes[row_idx, 0].imshow(raw_img)
    axes[row_idx, 0].set_title(f"Citra Asli\n{row['image_name']} ({raw_img.size})", fontsize=10)
    axes[row_idx, 0].axis('off')
    
    # 2. Citra Pasca-Preprocessing (Denormalized Tensor)
    proc_tensor, _, _ = fashion_dataset[data_idx]
    proc_img = denormalize(proc_tensor)
    axes[row_idx, 1].imshow(proc_img)
    axes[row_idx, 1].set_title(f"Setelah Preprocessing\nTensor [3, 224, 224]", fontsize=10)
    axes[row_idx, 1].axis('off')

plt.tight_layout()
plt.show()

## 6. Ringkasan & Verifikasi Kesiapan Preprocessing

In [ ]:
print("=" * 65)
print("🏆 LAPORAN STATUS PREPROCESSING CITRA")
print("=" * 65)
print(f"1. Dimensi Input Model     : (3, 224, 224) RGB Tensor")
print(f"2. Standar Normalisasi     : ImageNet Mean & Std")
print(f"3. Total Sampel Siap Olah  : {len(fashion_dataset):,} citra")
print(f"4. Konfigurasi DataLoader  : Batch Size = {BATCH_SIZE}, Total Batch = {len(fashion_loader)}")
print(f"5. Kesiapan Perangkat Keras: Terhubung ke {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print("-" * 65)
print("🎉 PIPELINE PREPROCESSING 100% SIAP UNTUK EKSTRAKSI FITUR DEEP LEARNING!")
print("=" * 65)